# Lab 4: A* Search — Operation Shaheen

| | |
|---|---|
| **Name** | Zohaib Hussain |
| **Roll No** | 23L-3087 |

---

Implement a complete A* Search algorithm to find the optimal interception path for a PAF drone.
All functions must be written by you from scratch.

**Zone entry costs:**

| Zone Code | Zone Type | Entry Cost |
|---|---|---|
| 0 | Open Sky | 1 |
| 1 | No-Fly Zone | Blocked (impassable) |
| 2 | Mountainous Terrain | 3 |
| 3 | Radar Blackout Zone | 2 |
| 4 | PAF Base | 0 |

> **Important:** `S` (start) and `G` (goal) are loaded as zone code `0` (Open Sky, entry cost `1`).  
> The **start cell's cost is not counted** — `g(start) = 0`. Total cost = sum of entry costs for all cells entered **after** the start.  
> Manhattan distance is the heuristic for this lab.

**When done:** Run all cells, download this file, rename it to your roll number (e.g. `21L-1234.ipynb`), and submit.

In [1]:
import heapq

grid1_content = """0  0  2  0  1  0  0
0  1  0  0  0  3  0
0  0  0  1  0  0  0
2  0  1  0  0  1  G
0  3  0  0  1  0  0
0  0  0  2  0  0  0
S  0  1  0  3  0  0"""

grid2_content = """S  0  0  2  0  1  0  0  3  0
0  1  0  0  3  0  0  1  0  0
0  0  2  0  0  0  1  0  0  0
0  1  0  1  0  2  0  0  1  0
0  0  0  0  0  0  3  0  0  0
2  0  1  0  1  0  0  0  2  0
0  3  0  0  0  1  0  1  0  0
0  0  0  1  0  0  0  0  3  0
1  0  2  0  0  1  0  2  0  0
0  0  0  3  0  0  4  0  1  G"""

with open('grid1.txt', 'w') as f: f.write(grid1_content)
with open('grid2.txt', 'w') as f: f.write(grid2_content)
print('Grid files saved.')

Grid files saved.


In [2]:
# Pre-written — do not modify.
#
# 'S' and 'G' tokens are replaced with zone code 0 (Open Sky, entry cost 1).
# The start cell's cost is never added to g — only cells entered AFTER
# the start contribute to the total cost.
def load_grid(filename):
    grid, start, goal = [], None, None
    with open(filename) as f:
        for r, line in enumerate(f):
            row = []
            for c, token in enumerate(line.split()):
                if   token == 'S': start = (r, c); row.append(0)
                elif token == 'G': goal  = (r, c); row.append(0)
                else: row.append(int(token))
            grid.append(row)
    return grid, start, goal

In [10]:
# Implement all five functions below.
# Do not modify the function signatures or docstrings.
import math

def heuristic(a, b):
    """Return the Manhattan distance between points a=(row,col) and b=(row,col)."""
    x1, y1 = a
    x2, y2 = b
    return math.sqrt(pow(x2 - x1, 2) 
                     + pow(y2 - y1, 2)
    )
    pass


zone_costs = {
    '0' : 1,
    '1' : None,
    '2' : 3,
    '3' : 2,
    '4' : 0
}

def get_zone_cost(grid, pos):
    """Return the entry cost of the cell at pos=(row, col).

    Zone code -> entry cost:
        0 (Open Sky)        -> 1
        1 (No-Fly Zone)     -> None  (impassable; caller must skip this cell)
        2 (Mountainous)     -> 3
        3 (Radar Blackout)  -> 2
        4 (PAF Base)        -> 0
    """
    x, y = pos
    if not grid or not grid[x][y]:
        return None

    return zone_costs[grid[x][y]]
    
    pass


def is_valid_pos(grid, row, col):
    if row < 0 or col < 0 or row >= len(grid) or col >= len(grid[0]):
        #print(f"Not Valid row {row} and col{col}")
        return False

    return True


def is_fly_zone(grid, row, col):
    if grid[row][col] == '1':
        return False
    return True
    
def get_neighbors(grid, pos):
    """Return a list of (row, col) tuples reachable from pos in one step.

    Rules:
    - 4-directional movement only (up, down, left, right).
    - Exclude cells that are out of bounds.
    - Exclude cells whose zone code is 1 (No-Fly Zone).
    """
    row, col = pos
    neighbors = []
    if not is_valid_pos(grid, row, col):
        return
    positions = [(-1, 0), # up
                (1, 0), # down
                (0, -1), # left
                (0, 1) # right
                ]
    for x,y in positions:
        if is_valid_pos(grid, row + x, col + y) and is_fly_zone(grid, row + x, col + y):
            neighbors.append(( row + x, col + y ))
    
    return neighbors
    
    pass

import heapq
def astar(grid, start, goal):
    """Find the least-cost path from start to goal using A*.

    - f(n) = g(n) + h(n)
    - Use a min-heap (heapq) as the open list.
    - Use a set as the closed list.
    - g(start) = 0  (the start cell's entry cost is NOT counted).
    - Return the path as a list of (row, col) tuples from start to goal,
      or None if no path exists.
    """
    openList = []
    closeList = set()
    pathTrace = {}
    # f(n), g(n), (row, col), (parentRow, parentCol) 
    heapq.heappush(openList, (0, 0, start, None) )

    while openList:
        ## expanding frontier from queue
        (fn, curr_cost, curr, curr_parent) = heapq.heappop(openList)

        ## we have check whether its already came out of heap
        if curr in closeList:
            continue # already visited
        ## store parent to reconstruct path
        pathTrace[curr] = curr_parent
            
        ## goal is found
        if curr == goal: 
            break
        closeList.add(curr)
        neighbors = get_neighbors(grid, curr)
        
        for neighbor in neighbors:
            g = curr_cost + get_zone_cost(grid, neighbor)
            h = heuristic(neighbor, goal)
            f = h + g

            # if not in openList or:
            heapq.heappush(openList, (f, g, neighbor, curr) )
    
    if goal not in pathTrace:
        return None

    ## reconstruct path
    path = []
    curr = goal

    while curr is not None:
        path.append(curr)
        curr = pathTrace[curr]

    path.reverse()
    return path 
                
            
            
        
    pass


def print_result(path, grid):
    """Print the path and its total cost.

    Output format:
        Path Found: (r,c) -> (r,c) -> ... -> (r,c)
        Total Cost: <integer>

    Total cost = sum of entry costs for every cell in the path EXCEPT the start.
    If path is None, print: No path found.
    """

    def print_result(path, grid):
        """Print the path and its total cost."""
    
        if path is None:
            print("No path found.")
            return
    
        # Print path
        path_str = " -> ".join(f"({r},{c})" for r, c in path)
        print(f"Path Found: {path_str}")
    
        # Calculate total cost (exclude start cell)
        total_cost = 0
        for cell in path[1:]:
            total_cost += get_zone_cost(grid, cell)
    
        print(f"Total Cost: {total_cost}")
    
    pass

In [11]:
# Expected output:
# Path Found: (6,0) -> (5,0) -> (5,1) -> (5,2) -> (4,2) -> (4,3) -> (3,3) -> (3,4) -> (2,4) -> (2,5) -> (2,6) -> (3,6)
# Total Cost: 11
print('--- grid1.txt ---')
grid, start, goal = load_grid('grid1.txt')
path = astar(grid, start, goal)
print_result(path, grid)

--- grid1.txt ---


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

In [12]:
# Expected output:
# Total Cost: 19
# Note: grid2 has several equally optimal paths. Your path may differ — only the total cost must be 19.
print('--- grid2.txt ---')
grid2, start2, goal2 = load_grid('grid2.txt')
path2 = astar(grid2, start2, goal2)
print_result(path2, grid2)

--- grid2.txt ---


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'